# Quick tutorial on Combosciplex.

## Prepare Notebook.

### Autoreload Environment.

In [ ]:
%reload_ext autoreload
%autoreload 2

### Import Libraries.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.preprocessing import OneHotEncoder

import sc_flow

## Data.

### Read data from disk.

In [ ]:
data_path = "/Users/lorenzo.consoli/Downloads/data/combosciplex.h5ad"

# create dataset
adata = sc.read_h5ad(data_path)
adata

### Visualize UMAP.

In [ ]:
sc.pl.embedding(adata, "umap", s=20, color=["source"], legend_loc="on data", frameon=False)

### Write One-hot drug representation.

In [ ]:
col1_vals = adata.obs["Drug1"]
col2_vals = adata.obs["Drug2"]

col_vals = np.concatenate((col1_vals, col2_vals), axis=0)
ohe = OneHotEncoder(sparse_output=False).fit(col_vals.reshape(-1, 1))

repr_dict = {}
for val in ohe.categories_[0]:
    val_arr = np.array([[val]], dtype=object)
    val_repr = ohe.transform(val_arr)
    repr_dict[val] = val_repr
adata.uns["drug_ohe"] = repr_dict

## Train Model.

### Register data.

In [ ]:
# register data and initialize model
sc_flow.SCFlow.register_adata(
    adata,
    sample_rep="X_pca",
    conditions={
        "drug": [
            "Drug1",
            "Drug2",
        ]
    },
    conditions_reps={"drug": "drug_ohe"},
    control_values_dict={"drug": "control"},
)

### Initialize Model.

In [ ]:
# register data and initialize model
model = sc_flow.SCFlow(
    method_id="cfm",
    vf_decoder_mlp_kwargs={
        "hidden_dims": [
            32,
            32,
        ]
    },
    condition_encoder_input_layers={
        "drug": {
            "hidden_dims": [16, 16],
            "output_dim": 4,
        }
    },
    device_id="mps",
    time_features_id="torch-cfm",
    conditioning_id="resnet1d",
    # match_fn=sc_flow.backends.torch.coupling.ot_linear_coupling
    # probability_path=probability_paths.SchrodingerBridgeProbabilityPath(1.0)
)

### Train model.

In [ ]:
# train model
model.train(adata, n_train_steps=150_000, optim_kwargs={"lr": 1e-4}, sort=True)

# display metrics
train_logs_df = model.trainer.get_train_logs_df()
train_logs_df.plot()

## Predict.

### Predict on input adata.

In [ ]:
pred_adata = model.predict(adata)
pred_adata

### Concatenate with real data.

In [ ]:
# write column for source
pred_adata.obs["source"] = "gen"
adata.obs["source"] = "real"

X_pca_real = adata.obsm["X_pca"]
X_pca_gen = pred_adata.X

X_pca = np.concat((X_pca_real, X_pca_gen), axis=0)

obs = pd.concat((adata.obs[["Drug1", "Drug2", "source"]], pred_adata.obs), axis=0)

adata_concat = sc.AnnData(X=X_pca, obs=obs, obsm={"X_pca": X_pca})

sc.pp.neighbors(adata_concat)
sc.tl.umap(adata_concat)

### Plot.

In [ ]:
sc.pl.embedding(
    adata_concat,
    "umap",
    # s=20,
    color=["source"],
    groups=["gen"],
    legend_loc="on data",
    frameon=False,
)